# Module 4: Modeling

**Suggested Pipeline**:

**Module 4.1: Baseline: TF-IDF + Logistic Regression**

**WHAT** we do:
- Convert text into TF-IDF features (1–2 grams)
- Train a Logistic Regression classifier

**WHY** we choose this model:
- Toxic comments often contain clear keywords, like swear words or insults. A TF-IDF model can already capture this kind of signal quite well.
- Logistic Regression is fast and stable. It also gives us a clean reference point before trying anything more complex.
- Another reason is interpretability. We can later check which words are driving the predictions.

**Module 4.2: Neural Model: BiLSTM**

**WHAT** we do:
- Use word embeddings
- Feed sequences into a BiLSTM model
- Output toxicity probability

**WHY** we choose this model:

A BiLSTM reads the sentence as a sequence. It can capture how words interact with each other. This helps with patterns like:
- negation
- sarcasm (to some extent)
- phrase-level toxicity

Another reason is generalization.
Even if the model is trained on English, it may learn some structure that transfers better to other languages.

**WHAT** we will expect:
- We may see improvement on harder cases.
- But the gain might not be large, because toxic signals are often very explicit.
- So we need to test if extra complexity really worth it.

**Module 4.3: Transformers: mBERT + XLM-R**

**WHAT** we do:
- Fine-tune mBERT on English training data
- Fine-tune XLM-R on the same data
- Evaluate both on multilingual validation/test

**WHY** we choose these models:
1. The earlier models are not designed for multiple languages. TF-IDF depends on exact words. BiLSTM depends on embeddings trained on English. Both are limited to other languages. They are both expecte to show a very poor generalization power on the val and test datasets.
2. mBERT is trained on many languages at the same time. It maps different languages into a shared space. This means even we train the model on English data, mBERT may still understand other languages.
3. XLM-R is a stronger version of this idea. It is trained on more data and usually learns better representations. In many tasks, it performs better than mBERT.

**WHAT** we will expect:

- mBERT should already perform better than previous models on non-English data.
- XLM-R is likely the best model overall.


## 4.1 Baseline: TF-IDF + Logistic Regression

In [ ]:
# Utility Functions
def print_section(title: str) -> None:
    """Print a formatted section title."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def evaluate_predictions(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = 0.5
) -> dict:
    """
    Evaluate prediction probabilities using a fixed threshold.

    Returns:
        A dictionary containing ROC-AUC, Precision, Recall, F1,
        confusion matrix, and positive prediction rate.
    """
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "positive_prediction_rate": float(np.mean(y_pred)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }
    return metrics


def print_metrics(metrics: dict, dataset_name: str) -> None:
    """Pretty-print evaluation metrics."""
    print_section(f"{dataset_name} - Evaluation Metrics")
    print(f"Threshold: {metrics['threshold']:.2f}")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"Positive Prediction Rate: {metrics['positive_prediction_rate']:.4f}")
    print("Confusion Matrix:")
    print(np.array(metrics["confusion_matrix"]))


def evaluate_by_language(
    df: pd.DataFrame,
    prob_col: str = "pred_prob",
    label_col: str = "toxic",
    lang_col: str = "lang",
    threshold: float = 0.5
) -> pd.DataFrame:
    """
    Compute evaluation metrics for each language separately.
    """
    results = []

    for lang, group in df.groupby(lang_col):
        y_true = group[label_col].values
        y_prob = group[prob_col].values
        y_pred = (y_prob >= threshold).astype(int)

        result = {
            "lang": lang,
            "n_samples": len(group),
            "positive_ratio": group[label_col].mean(),
            "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "positive_prediction_rate": np.mean(y_pred)
        }
        results.append(result)

    return pd.DataFrame(results).sort_values(by="roc_auc", ascending=False).reset_index(drop=True)


def get_top_features(
    pipeline: Pipeline,
    top_n: int = 20
) -> tuple:
    """
    Extract top positive and top negative TF-IDF features from the trained
    logistic regression model.
    """
    vectorizer = pipeline.named_steps["tfidf"]
    model = pipeline.named_steps["clf"]

    feature_names = np.array(vectorizer.get_feature_names_out())
    coefs = model.coef_[0]

    top_positive_idx = np.argsort(coefs)[-top_n:][::-1]
    top_negative_idx = np.argsort(coefs)[:top_n]

    top_positive = pd.DataFrame({
        "feature": feature_names[top_positive_idx],
        "coefficient": coefs[top_positive_idx]
    })

    top_negative = pd.DataFrame({
        "feature": feature_names[top_negative_idx],
        "coefficient": coefs[top_negative_idx]
    })

    return top_positive, top_negative


In [ ]:
# Load prepared data
print_section("Loading Prepared Data")

DATA_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/prepared data"

train_df = pd.read_csv(f"{DATA_PATH}/train_english_train.csv")
dev_df = pd.read_csv(f"{DATA_PATH}/train_english_dev.csv")
val_df = pd.read_csv(f"{DATA_PATH}/validation_clean.csv")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("Validation shape:", val_df.shape)


Loading Prepared Data
Train shape: (1887838, 3)
Dev shape: (209760, 3)
Validation shape: (7999, 4)


In [ ]:
# Prepare inputs
print_section("Preparing Inputs")

X_train = train_df["comment_text"].astype(str).values
y_train = train_df["toxic"].astype(int).values

X_dev = dev_df["comment_text"].astype(str).values
y_dev = dev_df["toxic"].astype(int).values

X_val = val_df["comment_text"].astype(str).values
y_val = val_df["toxic"].astype(int).values

print("Training positive ratio:", round(y_train.mean(), 4))
print("Dev positive ratio:", round(y_dev.mean(), 4))
print("Validation positive ratio:", round(y_val.mean(), 4))


Preparing Inputs
Training positive ratio: 0.0818
Dev positive ratio: 0.0818
Validation positive ratio: 0.1538


In [ ]:
# Build baseline pipeline
print_section("Building TF-IDF + Logistic Regression Pipeline")

baseline_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            max_features=100000,
            sublinear_tf=True
        )
    ),
    (
        "clf",
        LogisticRegression(
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])

print(baseline_pipeline)


Building TF-IDF + Logistic Regression Pipeline
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=100000, min_df=3,
                                 ngram_range=(1, 2), strip_accents='unicode',
                                 sublinear_tf=True)),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42, solver='liblinear'))])


In [ ]:
# Train model
print_section("Training Baseline Model")
baseline_pipeline.fit(X_train, y_train)
print("Training completed successfully.")


Training Baseline Model
Training completed successfully.


In [ ]:
# Predict probabilities
print_section("Generating Predictions")

dev_prob = baseline_pipeline.predict_proba(X_dev)[:, 1]
val_prob = baseline_pipeline.predict_proba(X_val)[:, 1]

print("Prediction generation completed.")


Generating Predictions
Prediction generation completed.


In [ ]:
# Evaluate on English dev set
dev_metrics = evaluate_predictions(
    y_true=y_dev,
    y_prob=dev_prob,
    threshold=0.5
)

print_metrics(dev_metrics, "English Dev Set")

print_section("English Dev Set - Classification Report")
print(classification_report(y_dev, (dev_prob >= 0.5).astype(int), digits=4, zero_division=0))


English Dev Set - Evaluation Metrics
Threshold: 0.50
ROC-AUC: 0.9456
Precision: 0.4616
Recall: 0.8281
F1 Score: 0.5927
Positive Prediction Rate: 0.1467
Confusion Matrix:
[[176037  16570]
 [  2949  14204]]

English Dev Set - Classification Report
              precision    recall  f1-score   support

           0     0.9835    0.9140    0.9475    192607
           1     0.4616    0.8281    0.5927     17153

    accuracy                         0.9069    209760
   macro avg     0.7225    0.8710    0.7701    209760
weighted avg     0.9408    0.9069    0.9185    209760



In [ ]:
# Evaluate on multilingual validation set
val_metrics = evaluate_predictions(
    y_true=y_val,
    y_prob=val_prob,
    threshold=0.5
)

print_metrics(val_metrics, "Multilingual Validation Set")

print_section("Multilingual Validation Set - Classification Report")
print(classification_report(y_val, (val_prob >= 0.5).astype(int), digits=4, zero_division=0))


Multilingual Validation Set - Evaluation Metrics
Threshold: 0.50
ROC-AUC: 0.6223
Precision: 0.3924
Recall: 0.0756
F1 Score: 0.1268
Positive Prediction Rate: 0.0296
Confusion Matrix:
[[6625  144]
 [1137   93]]

Multilingual Validation Set - Classification Report
              precision    recall  f1-score   support

           0     0.8535    0.9787    0.9118      6769
           1     0.3924    0.0756    0.1268      1230

    accuracy                         0.8399      7999
   macro avg     0.6230    0.5272    0.5193      7999
weighted avg     0.7826    0.8399    0.7911      7999



In [ ]:
# Per-language evaluation on validation set
print_section("Per-Language Evaluation on Validation Set")

val_eval_df = val_df.copy()
val_eval_df["pred_prob"] = val_prob

per_lang_results = evaluate_by_language(
    df=val_eval_df,
    prob_col="pred_prob",
    label_col="toxic",
    lang_col="lang",
    threshold=0.5
)

print(per_lang_results)


Per-Language Evaluation on Validation Set
  lang  n_samples  positive_ratio   roc_auc  precision    recall        f1  \
0   tr       2999        0.106702  0.660836   0.560000  0.087500  0.151351   
1   it       2500        0.195200  0.615092   0.386364  0.034836  0.063910   
2   es       2500        0.168800  0.586452   0.335664  0.113744  0.169912   

   positive_prediction_rate  
0                  0.016672  
1                  0.017600  
2                  0.057200  


In [ ]:
# Inspect top features
print_section("Top Positive and Negative Features")

top_positive_features, top_negative_features = get_top_features(
    baseline_pipeline,
    top_n=20
)

print("Top Positive Features (associated with toxic class):")
print(top_positive_features)

print("\nTop Negative Features (associated with non-toxic class):")
print(top_negative_features)



Top Positive and Negative Features
Top Positive Features (associated with toxic class):
       feature  coefficient
0       stupid    49.420431
1       idiots    37.937425
2        idiot    36.950113
3    stupidity    35.507920
4      idiotic    30.382139
5     pathetic    29.432082
6     ignorant    28.825394
7         crap    27.994857
8         dumb    27.406899
9    hypocrite    26.872973
10  ridiculous    26.457238
11        damn    25.859994
12       fools    25.663885
13        shit    24.159069
14     foolish    24.129086
15      morons    23.910959
16       moron    23.470759
17  hypocrites    23.367008
18        fool    23.270945
19      idiocy    22.381973

Top Negative Features (associated with non-toxic class):
        feature  coefficient
0       fool me    -7.683949
1     knee jerk    -6.130786
2       to fool    -5.659956
3      can fool    -5.451304
4     give damn    -4.903262
5   ignorant of    -4.572944
6   pretty damn    -4.264640
7   white house    -4.241712
8   

In [ ]:
# Save predictions and metrics for later modules
print_section("Saving Outputs")

SAVE_BASELINE_OUTPUTS = True
OUTPUT_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs"

if SAVE_BASELINE_OUTPUTS:
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    # Save validation predictions
    dev_output = dev_df.copy()
    dev_output["pred_prob_lr"] = dev_prob
    dev_output["pred_label_lr_05"] = (dev_prob >= 0.5).astype(int)
    dev_output.to_csv(f"{OUTPUT_PATH}/lr_dev_predictions.csv", index=False)

    val_output = val_df.copy()
    val_output["pred_prob_lr"] = val_prob
    val_output["pred_label_lr_05"] = (val_prob >= 0.5).astype(int)
    val_output.to_csv(f"{OUTPUT_PATH}/lr_validation_predictions.csv", index=False)

    # Save metrics
    metrics_dict = {
        "english_dev": dev_metrics,
        "multilingual_validation": val_metrics
    }

    with open(f"{OUTPUT_PATH}/lr_metrics.json", "w") as f:
        json.dump(metrics_dict, f, indent=4)

    per_lang_results.to_csv(f"{OUTPUT_PATH}/lr_per_language_results.csv", index=False)
    top_positive_features.to_csv(f"{OUTPUT_PATH}/lr_top_positive_features.csv", index=False)
    top_negative_features.to_csv(f"{OUTPUT_PATH}/lr_top_negative_features.csv", index=False)

    print(f"Outputs saved to: {OUTPUT_PATH}")


Saving Outputs
Outputs saved to: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs


In [ ]:
# Final summary
print_section("Module 4.1 Final Summary")

summary_table = pd.DataFrame([
    {
        "dataset": "english_dev",
        "roc_auc": round(dev_metrics["roc_auc"], 4),
        "precision": round(dev_metrics["precision"], 4),
        "recall": round(dev_metrics["recall"], 4),
        "f1": round(dev_metrics["f1"], 4),
        "positive_prediction_rate": round(dev_metrics["positive_prediction_rate"], 4)
    },
    {
        "dataset": "multilingual_validation",
        "roc_auc": round(val_metrics["roc_auc"], 4),
        "precision": round(val_metrics["precision"], 4),
        "recall": round(val_metrics["recall"], 4),
        "f1": round(val_metrics["f1"], 4),
        "positive_prediction_rate": round(val_metrics["positive_prediction_rate"], 4)
    }
])

print(summary_table)


Module 4.1 Final Summary
                   dataset  roc_auc  precision  recall      f1  \
0              english_dev   0.9456     0.4616  0.8281  0.5927   
1  multilingual_validation   0.6223     0.3924  0.0756  0.1268   

   positive_prediction_rate  
0                    0.1467  
1                    0.0296  


**Discoveries**:
- TF-IDF performs well on English dev dataset
- However, it fails to generalize its prediction power to validation set with multiple languages. Recall = .0756 meaning only 1/13 toxic comments are detected.
- Among all foreign languages, spanish is the best, while italian is the worst.
- Top positive features include many swaer words (stupid, idiot, crap, dumb, etc.)

## 4.2 BiLSTM

**Goal**: To test if context/sequence info can improve zero-shot multilingal performance.

In [ ]:
# Reproductability + Utility Functions
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

def print_section(title: str) -> None:
    """Print a formatted section title."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def evaluate_predictions(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = 0.5
) -> dict:
    """
    Evaluate prediction probabilities using a fixed threshold.
    """
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "positive_prediction_rate": float(np.mean(y_pred)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }
    return metrics


def print_metrics(metrics: dict, dataset_name: str) -> None:
    """Pretty-print evaluation metrics."""
    print_section(f"{dataset_name} - Evaluation Metrics")
    print(f"Threshold: {metrics['threshold']:.2f}")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"Positive Prediction Rate: {metrics['positive_prediction_rate']:.4f}")
    print("Confusion Matrix:")
    print(np.array(metrics["confusion_matrix"]))


def evaluate_by_language(
    df: pd.DataFrame,
    prob_col: str = "pred_prob",
    label_col: str = "toxic",
    lang_col: str = "lang",
    threshold: float = 0.5
) -> pd.DataFrame:
    """
    Compute evaluation metrics for each language separately.
    """
    results = []

    for lang, group in df.groupby(lang_col):
        y_true = group[label_col].values
        y_prob = group[prob_col].values
        y_pred = (y_prob >= threshold).astype(int)

        result = {
            "lang": lang,
            "n_samples": len(group),
            "positive_ratio": float(group[label_col].mean()),
            "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "positive_prediction_rate": float(np.mean(y_pred))
        }
        results.append(result)

    return pd.DataFrame(results).sort_values(by="roc_auc", ascending=False).reset_index(drop=True)


def get_class_weight_dict(y: np.ndarray) -> dict:
    """
    Create class weights for imbalanced binary classification.
    """
    n_total = len(y)
    n_pos = np.sum(y == 1)
    n_neg = np.sum(y == 0)

    weight_for_0 = n_total / (2.0 * n_neg)
    weight_for_1 = n_total / (2.0 * n_pos)

    return {
        0: float(weight_for_0),
        1: float(weight_for_1)
    }

In [ ]:
# Load prepared data
print_section("Loading Prepared Data")

DATA_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/prepared data"

train_df = pd.read_csv(f"{DATA_PATH}/train_english_train.csv")
dev_df = pd.read_csv(f"{DATA_PATH}/train_english_dev.csv")
val_df = pd.read_csv(f"{DATA_PATH}/validation_clean.csv")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("Validation shape:", val_df.shape)


Loading Prepared Data
Train shape: (1887838, 3)
Dev shape: (209760, 3)
Validation shape: (7999, 4)


In [ ]:
# Prepare text and labels
print_section("Preparing Text and Labels")

X_train_text = train_df["comment_text"].astype(str).tolist()
y_train = train_df["toxic"].astype(int).values

X_dev_text = dev_df["comment_text"].astype(str).tolist()
y_dev = dev_df["toxic"].astype(int).values

X_val_text = val_df["comment_text"].astype(str).tolist()
y_val = val_df["toxic"].astype(int).values

print("Training positive ratio:", round(y_train.mean(), 4))
print("Dev positive ratio:", round(y_dev.mean(), 4))
print("Validation positive ratio:", round(y_val.mean(), 4))


Preparing Text and Labels
Training positive ratio: 0.0818
Dev positive ratio: 0.0818
Validation positive ratio: 0.1538


In [ ]:
# Tokenization and sequence preparation
print_section("Tokenization and Sequence Preparation")

MAX_NUM_WORDS = 50000
MAX_SEQUENCE_LENGTH = 128
OOV_TOKEN = "<OOV>"

tokenizer = Tokenizer(
    num_words=MAX_NUM_WORDS,
    oov_token=OOV_TOKEN
)

# Important:
# Fit tokenizer on English training data only.
# This avoids using validation information during training preparation.
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_dev_seq = tokenizer.texts_to_sequences(X_dev_text)
X_val_seq = tokenizer.texts_to_sequences(X_val_text)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_dev_pad = pad_sequences(
    X_dev_seq,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

vocab_size = min(MAX_NUM_WORDS, len(tokenizer.word_index) + 1)

print("Vocabulary size used:", vocab_size)
print("Train padded shape:", X_train_pad.shape)
print("Dev padded shape:", X_dev_pad.shape)
print("Validation padded shape:", X_val_pad.shape)


Tokenization and Sequence Preparation
Vocabulary size used: 50000
Train padded shape: (1887838, 128)
Dev padded shape: (209760, 128)
Validation padded shape: (7999, 128)


In [ ]:
# Build BiLSTM model
print_section("Building BiLSTM Model")

EMBEDDING_DIM = 128
LSTM_UNITS = 64
DROPOUT_RATE = 0.3
LEARNING_RATE = 1e-3

bilstm_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH
    ),
    Bidirectional(
        LSTM(
            LSTM_UNITS,
            return_sequences=False,
            dropout=0.2,
            recurrent_dropout=0.0
        )
    ),
    Dropout(DROPOUT_RATE),
    Dense(64, activation="relu"),
    Dropout(DROPOUT_RATE),
    Dense(1, activation="sigmoid")
])

bilstm_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_model.summary()


Building BiLSTM Model


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train model
print_section("Training BiLSTM Model")

BATCH_SIZE = 256
EPOCHS = 5

class_weight_dict = get_class_weight_dict(y_train)
print("Class weights:", class_weight_dict)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

history = bilstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_dev_pad, y_dev),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=[early_stopping],
    verbose=1
)

# Generate predictions
print_section("Generating Predictions")

dev_prob = bilstm_model.predict(X_dev_pad, batch_size=1024, verbose=1).reshape(-1)
val_prob = bilstm_model.predict(X_val_pad, batch_size=1024, verbose=1).reshape(-1)

print("Prediction generation completed.")


Training BiLSTM Model
Class weights: {0: 0.5445298037390025, 1: 6.114217423128494}
Epoch 1/5
7375/7375 ━━━━━━━━━━━━━━━━━━━━ 126s 16ms/step - accuracy: 0.8657 - loss: 0.3111 - val_accuracy: 0.8486 - val_loss: 0.3261
Epoch 2/5
7375/7375 ━━━━━━━━━━━━━━━━━━━━ 118s 16ms/step - accuracy: 0.8818 - loss: 0.2563 - val_accuracy: 0.8594 - val_loss: 0.2964
Epoch 3/5
7375/7375 ━━━━━━━━━━━━━━━━━━━━ 116s 16ms/step - accuracy: 0.8885 - loss: 0.2297 - val_accuracy: 0.8657 - val_loss: 0.2920
Epoch 4/5
7375/7375 ━━━━━━━━━━━━━━━━━━━━ 120s 16ms/step - accuracy: 0.8963 - loss: 0.2069 - val_accuracy: 0.8665 - val_loss: 0.2785
Epoch 5/5
7375/7375 ━━━━━━━━━━━━━━━━━━━━ 116s 16ms/step - accuracy: 0.9053 - loss: 0.1868 - val_accuracy: 0.8606 - val_loss: 0.3087
Restoring model weights from the end of the best epoch: 4.

Generating Predictions
205/205 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
Prediction generation completed.


In [ ]:
# Evaluate on English dev set
dev_metrics = evaluate_predictions(
    y_true=y_dev,
    y_prob=dev_prob,
    threshold=0.5
)

print_metrics(dev_metrics, "English Dev Set")

print_section("English Dev Set - Classification Report")
print(classification_report(y_dev, (dev_prob >= 0.5).astype(int), digits=4, zero_division=0))


English Dev Set - Evaluation Metrics
Threshold: 0.50
ROC-AUC: 0.9482
Precision: 0.3697
Recall: 0.8968
F1 Score: 0.5236
Positive Prediction Rate: 0.1984
Confusion Matrix:
[[166383  26224]
 [  1770  15383]]

English Dev Set - Classification Report
              precision    recall  f1-score   support

           0     0.9895    0.8638    0.9224    192607
           1     0.3697    0.8968    0.5236     17153

    accuracy                         0.8665    209760
   macro avg     0.6796    0.8803    0.7230    209760
weighted avg     0.9388    0.8665    0.8898    209760



In [ ]:
# Evaluate on multilingual validation set
val_metrics = evaluate_predictions(
    y_true=y_val,
    y_prob=val_prob,
    threshold=0.5
)

print_metrics(val_metrics, "Multilingual Validation Set")

print_section("Multilingual Validation Set - Classification Report")
print(classification_report(y_val, (val_prob >= 0.5).astype(int), digits=4, zero_division=0))


Multilingual Validation Set - Evaluation Metrics
Threshold: 0.50
ROC-AUC: 0.5679
Precision: 0.2587
Recall: 0.2057
F1 Score: 0.2292
Positive Prediction Rate: 0.1223
Confusion Matrix:
[[6044  725]
 [ 977  253]]

Multilingual Validation Set - Classification Report
              precision    recall  f1-score   support

           0     0.8608    0.8929    0.8766      6769
           1     0.2587    0.2057    0.2292      1230

    accuracy                         0.7872      7999
   macro avg     0.5598    0.5493    0.5529      7999
weighted avg     0.7683    0.7872    0.7770      7999



In [ ]:
# Per-language evaluation on validation set
print_section("Per-Language Evaluation on Validation Set")

val_eval_df = val_df.copy()
val_eval_df["pred_prob"] = val_prob

per_lang_results = evaluate_by_language(
    df=val_eval_df,
    prob_col="pred_prob",
    label_col="toxic",
    lang_col="lang",
    threshold=0.5
)

print(per_lang_results)


Per-Language Evaluation on Validation Set
  lang  n_samples  positive_ratio   roc_auc  precision    recall        f1  \
0   es       2500        0.168800  0.641660   0.266402  0.317536  0.289730   
1   tr       2999        0.106702  0.557773   0.286957  0.103125  0.151724   
2   it       2500        0.195200  0.524580   0.238889  0.176230  0.202830   

   positive_prediction_rate  
0                  0.201200  
1                  0.038346  
2                  0.144000  


In [ ]:
# Save outputs for later modules
print_section("Saving Outputs")

SAVE_BILSTM_OUTPUTS = True
OUTPUT_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs"

if SAVE_BILSTM_OUTPUTS:
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    dev_output = dev_df.copy()
    dev_output["pred_prob_bilstm"] = dev_prob
    dev_output["pred_label_bilstm_05"] = (dev_prob >= 0.5).astype(int)
    dev_output.to_csv(f"{OUTPUT_PATH}/bilstm_dev_predictions.csv", index=False)

    val_output = val_df.copy()
    val_output["pred_prob_bilstm"] = val_prob
    val_output["pred_label_bilstm_05"] = (val_prob >= 0.5).astype(int)
    val_output.to_csv(f"{OUTPUT_PATH}/bilstm_validation_predictions.csv", index=False)

    metrics_dict = {
        "english_dev": dev_metrics,
        "multilingual_validation": val_metrics
    }

    with open(f"{OUTPUT_PATH}/bilstm_metrics.json", "w") as f:
        json.dump(metrics_dict, f, indent=4)

    per_lang_results.to_csv(f"{OUTPUT_PATH}/bilstm_per_language_results.csv", index=False)

    print(f"Outputs saved to: {OUTPUT_PATH}")


Saving Outputs
Outputs saved to: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs


In [ ]:
# Final summary
print_section("Module 4.2 Final Summary")

summary_table = pd.DataFrame([
    {
        "dataset": "english_dev",
        "roc_auc": round(dev_metrics["roc_auc"], 4),
        "precision": round(dev_metrics["precision"], 4),
        "recall": round(dev_metrics["recall"], 4),
        "f1": round(dev_metrics["f1"], 4),
        "positive_prediction_rate": round(dev_metrics["positive_prediction_rate"], 4)
    },
    {
        "dataset": "multilingual_validation",
        "roc_auc": round(val_metrics["roc_auc"], 4),
        "precision": round(val_metrics["precision"], 4),
        "recall": round(val_metrics["recall"], 4),
        "f1": round(val_metrics["f1"], 4),
        "positive_prediction_rate": round(val_metrics["positive_prediction_rate"], 4)
    }
])

print(summary_table)


Module 4.2 Final Summary
                   dataset  roc_auc  precision  recall      f1  \
0              english_dev   0.9482     0.3697  0.8968  0.5236   
1  multilingual_validation   0.5679     0.2587  0.2057  0.2292   

   positive_prediction_rate  
0                    0.1984  
1                    0.1223  


## 4.3 Transformer-based models

### 4.3.1 mBERT (Please take the **DEBUG** version, ignore this one)

In [ ]:
# Reproductability & Utility Functions
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
def print_section(title: str) -> None:
    """Print a formatted section title."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def evaluate_predictions(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = 0.5
) -> dict:
    """
    Evaluate prediction probabilities using a fixed threshold.
    """
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "positive_prediction_rate": float(np.mean(y_pred)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }
    return metrics


def print_metrics(metrics: dict, dataset_name: str) -> None:
    """Pretty-print evaluation metrics."""
    print_section(f"{dataset_name} - Evaluation Metrics")
    print(f"Threshold: {metrics['threshold']:.2f}")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"Positive Prediction Rate: {metrics['positive_prediction_rate']:.4f}")
    print("Confusion Matrix:")
    print(np.array(metrics["confusion_matrix"]))


def evaluate_by_language(
    df: pd.DataFrame,
    prob_col: str = "pred_prob",
    label_col: str = "toxic",
    lang_col: str = "lang",
    threshold: float = 0.5
) -> pd.DataFrame:
    """
    Compute evaluation metrics for each language separately.
    """
    results = []

    for lang, group in df.groupby(lang_col):
        y_true = group[label_col].values
        y_prob = group[prob_col].values
        y_pred = (y_prob >= threshold).astype(int)

        result = {
            "lang": lang,
            "n_samples": len(group),
            "positive_ratio": float(group[label_col].mean()),
            "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "positive_prediction_rate": float(np.mean(y_pred))
        }
        results.append(result)

    return pd.DataFrame(results).sort_values(by="roc_auc", ascending=False).reset_index(drop=True)


def sigmoid_numpy(x: np.ndarray) -> np.ndarray:
    """Apply sigmoid to logits."""
    return 1.0 / (1.0 + np.exp(-x))



In [ ]:
# Load prepared data
print_section("Loading Prepared Data")

DATA_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/prepared data"

train_df = pd.read_csv(f"{DATA_PATH}/train_english_train.csv")
dev_df = pd.read_csv(f"{DATA_PATH}/train_english_dev.csv")
val_df = pd.read_csv(f"{DATA_PATH}/validation_clean.csv")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("Validation shape:", val_df.shape)

# Optional sampling for faster experimentation
USE_FAST_DEBUG = False

if USE_FAST_DEBUG:
    train_df = train_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)
    dev_df = dev_df.sample(frac=0.2, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(frac=0.2, random_state=SEED).reset_index(drop=True)

print("Final train shape:", train_df.shape)
print("Final dev shape:", dev_df.shape)
print("Final validation shape:", val_df.shape)


Loading Prepared Data
Train shape: (1887838, 3)
Dev shape: (209760, 3)
Validation shape: (7999, 4)
Final train shape: (1887838, 3)
Final dev shape: (209760, 3)
Final validation shape: (7999, 4)


In [ ]:
# Prepare Hugging Face datasets
print_section("Preparing Datasets")

train_df = train_df[["comment_text", "toxic"]].copy()
dev_df = dev_df[["comment_text", "toxic"]].copy()
val_df = val_df[["comment_text", "toxic", "lang"]].copy()

train_df["label"] = train_df["toxic"].astype(int)
dev_df["label"] = dev_df["toxic"].astype(int)
val_df["label"] = val_df["toxic"].astype(int)

train_dataset = Dataset.from_pandas(train_df[["comment_text", "label"]], preserve_index=False)
dev_dataset = Dataset.from_pandas(dev_df[["comment_text", "label"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["comment_text", "label", "lang"]], preserve_index=False)

print(train_dataset)
print(dev_dataset)
print(val_dataset)


Preparing Datasets
Dataset({
    features: ['comment_text', 'label'],
    num_rows: 1887838
})
Dataset({
    features: ['comment_text', 'label'],
    num_rows: 209760
})
Dataset({
    features: ['comment_text', 'label', 'lang'],
    num_rows: 7999
})


In [ ]:
# Tokenizer and tokenization
print_section("Loading Tokenizer")

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
dev_dataset = dev_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)



Loading Tokenizer


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1887838 [00:00<?, ? examples/s]

Map:   0%|          | 0/209760 [00:00<?, ? examples/s]

Map:   0%|          | 0/7999 [00:00<?, ? examples/s]

In [ ]:
# Class weights for imbalanced training
print_section("Computing Class Weights")

train_labels = train_df["label"].values
n_total = len(train_labels)
n_pos = np.sum(train_labels == 1)
n_neg = np.sum(train_labels == 0)

weight_for_0 = n_total / (2.0 * n_neg)
weight_for_1 = n_total / (2.0 * n_pos)

class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float)

print("Class weights:", class_weights.tolist())



Computing Class Weights
Class weights: [0.5445297956466675, 6.114217281341553]


In [ ]:
# Custom Trainer with weighted loss
print_section("Building Model")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="single_label_classification"
)

class WeightedBCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits.view(-1)

        loss_fn = torch.nn.BCEWithLogitsLoss(
            pos_weight=class_weights[1].to(logits.device)
        )
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


Building Model


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Training arguments
print_section("Setting Training Arguments")

OUTPUT_DIR = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/mbert_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Core training setup
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,

    # Evaluation / saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,

    # Runtime
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



Setting Training Arguments


In [ ]:
# Build trainer
print_section("Initializing Trainer")

trainer = WeightedBCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# Train model
print_section("Training mBERT")
trainer.train()
print("Training completed successfully.")


Initializing Trainer

Training mBERT


Epoch,Training Loss,Validation Loss
1,1.021666,0.928546
2,0.849709,0.927772


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Training completed successfully.


In [ ]:
# Predict on English dev
print_section("Predicting on English Dev Set")

dev_predictions = trainer.predict(dev_dataset)
dev_logits = dev_predictions.predictions.reshape(-1)
dev_prob = sigmoid_numpy(dev_logits)
print("dev_logits summary:")
print("min:", np.min(dev_logits))
print("max:", np.max(dev_logits))
print("mean:", np.mean(dev_logits))
print("std:", np.std(dev_logits))

print("\ndev_prob summary:")
print("min:", np.min(dev_prob))
print("max:", np.max(dev_prob))
print("mean:", np.mean(dev_prob))
print("std:", np.std(dev_prob))

print("\nFirst 20 dev_prob:")
print(dev_prob[:20])
y_dev = dev_df["label"].values

dev_metrics = evaluate_predictions(
    y_true=y_dev,
    y_prob=dev_prob,
    threshold=0.5
)

print_metrics(dev_metrics, "English Dev Set")

print_section("English Dev Set - Classification Report")
print(classification_report(y_dev, (dev_prob >= 0.5).astype(int), digits=4, zero_division=0))


Predicting on English Dev Set


Epoch,Training Loss,Validation Loss


dev_logits summary:
min: -0.81103516
max: -0.8105469
mean: -0.8105511
std: 4.5148903e-05

dev_prob summary:
min: 0.30766994
max: 0.30777398
mean: 0.3077731
std: 9.620101e-06

First 20 dev_prob:
[0.30777398 0.30777398 0.30777398 0.30777398 0.30777398 0.30777398
 0.30777398 0.30777398 0.30777398 0.30777398 0.30777398 0.30777398
 0.30777398 0.30777398 0.30777398 0.30777398 0.30777398 0.30777398
 0.30777398 0.30777398]

English Dev Set - Evaluation Metrics
Threshold: 0.50
ROC-AUC: 0.5000
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
Positive Prediction Rate: 0.0000
Confusion Matrix:
[[192607      0]
 [ 17153      0]]

English Dev Set - Classification Report
              precision    recall  f1-score   support

           0     0.9182    1.0000    0.9574    192607
           1     0.0000    0.0000    0.0000     17153

    accuracy                         0.9182    209760
   macro avg     0.4591    0.5000    0.4787    209760
weighted avg     0.8431    0.9182    0.8791    209760



In [ ]:
# Predict on multilingual validation
print_section("Predicting on Multilingual Validation Set")

val_predictions = trainer.predict(val_dataset)
val_logits = val_predictions.predictions.reshape(-1)
val_prob = sigmoid_numpy(val_logits)
y_val = val_df["label"].values

val_metrics = evaluate_predictions(
    y_true=y_val,
    y_prob=val_prob,
    threshold=0.5
)

print_metrics(val_metrics, "Multilingual Validation Set")

print_section("Multilingual Validation Set - Classification Report")
print(classification_report(y_val, (val_prob >= 0.5).astype(int), digits=4, zero_division=0))


Predicting on Multilingual Validation Set



Multilingual Validation Set - Evaluation Metrics
Threshold: 0.50
ROC-AUC: 0.5000
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
Positive Prediction Rate: 0.0000
Confusion Matrix:
[[6769    0]
 [1230    0]]

Multilingual Validation Set - Classification Report
              precision    recall  f1-score   support

           0     0.8462    1.0000    0.9167      6769
           1     0.0000    0.0000    0.0000      1230

    accuracy                         0.8462      7999
   macro avg     0.4231    0.5000    0.4584      7999
weighted avg     0.7161    0.8462    0.7757      7999



In [ ]:
# Per-language evaluation on validation set
print_section("Per-Language Evaluation on Validation Set")

val_eval_df = val_df.copy()
val_eval_df["pred_prob"] = val_prob

per_lang_results = evaluate_by_language(
    df=val_eval_df,
    prob_col="pred_prob",
    label_col="toxic",
    lang_col="lang",
    threshold=0.5
)

print(per_lang_results)


Per-Language Evaluation on Validation Set
  lang  n_samples  positive_ratio  roc_auc  precision  recall   f1  \
0   es       2500        0.168800      0.5        0.0     0.0  0.0   
1   it       2500        0.195200      0.5        0.0     0.0  0.0   
2   tr       2999        0.106702      0.5        0.0     0.0  0.0   

   positive_prediction_rate  
0                       0.0  
1                       0.0  
2                       0.0  


In [ ]:
# Save outputs for later modules
print_section("Saving Outputs")

SAVE_MBERT_OUTPUTS = True
OUTPUT_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs"

if SAVE_MBERT_OUTPUTS:
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    # Save trained model and tokenizer
    trainer.save_model(f"{OUTPUT_PATH}/mbert_best_model")
    tokenizer.save_pretrained(f"{OUTPUT_PATH}/mbert_best_model")

    # Save dev predictions
    dev_output = dev_df.copy()
    dev_output["pred_prob_mbert"] = dev_prob
    dev_output["pred_label_mbert_05"] = (dev_prob >= 0.5).astype(int)
    dev_output.to_csv(f"{OUTPUT_PATH}/mbert_dev_predictions.csv", index=False)

    # Save validation predictions
    val_output = val_df.copy()
    val_output["pred_prob_mbert"] = val_prob
    val_output["pred_label_mbert_05"] = (val_prob >= 0.5).astype(int)
    val_output.to_csv(f"{OUTPUT_PATH}/mbert_validation_predictions.csv", index=False)

    # Save metrics
    metrics_dict = {
        "english_dev": dev_metrics,
        "multilingual_validation": val_metrics
    }

    with open(f"{OUTPUT_PATH}/mbert_metrics.json", "w") as f:
        json.dump(metrics_dict, f, indent=4)

    per_lang_results.to_csv(f"{OUTPUT_PATH}/mbert_per_language_results.csv", index=False)

    print(f"Outputs saved to: {OUTPUT_PATH}")


Saving Outputs


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Outputs saved to: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs


In [ ]:
# Final summary
print_section("Module 4.3.1 Final Summary")

summary_table = pd.DataFrame([
    {
        "dataset": "english_dev",
        "roc_auc": round(dev_metrics["roc_auc"], 4),
        "precision": round(dev_metrics["precision"], 4),
        "recall": round(dev_metrics["recall"], 4),
        "f1": round(dev_metrics["f1"], 4),
        "positive_prediction_rate": round(dev_metrics["positive_prediction_rate"], 4)
    },
    {
        "dataset": "multilingual_validation",
        "roc_auc": round(val_metrics["roc_auc"], 4),
        "precision": round(val_metrics["precision"], 4),
        "recall": round(val_metrics["recall"], 4),
        "f1": round(val_metrics["f1"], 4),
        "positive_prediction_rate": round(val_metrics["positive_prediction_rate"], 4)
    }
])

print(summary_table)


Module 4.3.1 Final Summary
                   dataset  roc_auc  precision  recall   f1  \
0              english_dev      0.5        0.0     0.0  0.0   
1  multilingual_validation      0.5        0.0     0.0  0.0   

   positive_prediction_rate  
0                       0.0  
1                       0.0  


#### Debug

In [ ]:
# ============================================================
# Module 4.3.1: Core Model 1
# mBERT (2-class stable version) for Zero-Shot Cross-Lingual Toxic Comment Detection
# ============================================================

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# 2. Utility functions
# ------------------------------------------------------------
def print_section(title: str) -> None:
    """Print a formatted section title."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def evaluate_predictions(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = 0.5
) -> dict:
    """
    Evaluate prediction probabilities using a fixed threshold.
    """
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "positive_prediction_rate": float(np.mean(y_pred)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }
    return metrics


def print_metrics(metrics: dict, dataset_name: str) -> None:
    """Pretty-print evaluation metrics."""
    print_section(f"{dataset_name} - Evaluation Metrics")
    print(f"Threshold: {metrics['threshold']:.2f}")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"Positive Prediction Rate: {metrics['positive_prediction_rate']:.4f}")
    print("Confusion Matrix:")
    print(np.array(metrics["confusion_matrix"]))


def evaluate_by_language(
    df: pd.DataFrame,
    prob_col: str = "pred_prob",
    label_col: str = "toxic",
    lang_col: str = "lang",
    threshold: float = 0.5
) -> pd.DataFrame:
    """
    Compute evaluation metrics for each language separately.
    """
    results = []

    for lang, group in df.groupby(lang_col):
        y_true = group[label_col].values
        y_prob = group[prob_col].values
        y_pred = (y_prob >= threshold).astype(int)

        result = {
            "lang": lang,
            "n_samples": len(group),
            "positive_ratio": float(group[label_col].mean()),
            "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "positive_prediction_rate": float(np.mean(y_pred))
        }
        results.append(result)

    return pd.DataFrame(results).sort_values(by="roc_auc", ascending=False).reset_index(drop=True)



In [ ]:
# ------------------------------------------------------------
# 3. Load prepared data
# ------------------------------------------------------------
print_section("Loading Prepared Data")

DATA_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/prepared data"

train_df = pd.read_csv(f"{DATA_PATH}/train_english_train.csv")
dev_df = pd.read_csv(f"{DATA_PATH}/train_english_dev.csv")
val_df = pd.read_csv(f"{DATA_PATH}/validation_clean.csv")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("Validation shape:", val_df.shape)


# ------------------------------------------------------------
# 4. Optional sampling for faster experimentation
# ------------------------------------------------------------
# Set USE_FAST_DEBUG = True if you want a smaller run first.
# For the real run, keep it False.

USE_FAST_DEBUG = False

if USE_FAST_DEBUG:
    train_df = train_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)
    dev_df = dev_df.sample(frac=0.2, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(frac=0.2, random_state=SEED).reset_index(drop=True)

print("Final train shape:", train_df.shape)
print("Final dev shape:", dev_df.shape)
print("Final validation shape:", val_df.shape)


# ------------------------------------------------------------
# 5. Prepare Hugging Face datasets
# ------------------------------------------------------------
print_section("Preparing Datasets")

train_df = train_df[["comment_text", "toxic"]].copy()
dev_df = dev_df[["comment_text", "toxic"]].copy()
val_df = val_df[["comment_text", "toxic", "lang"]].copy()

train_df["label"] = train_df["toxic"].astype(int)
dev_df["label"] = dev_df["toxic"].astype(int)
val_df["label"] = val_df["toxic"].astype(int)

train_dataset = Dataset.from_pandas(train_df[["comment_text", "label"]], preserve_index=False)
dev_dataset = Dataset.from_pandas(dev_df[["comment_text", "label"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["comment_text", "label", "lang"]], preserve_index=False)

print(train_dataset)
print(dev_dataset)
print(val_dataset)





Loading Prepared Data
Train shape: (1887838, 3)
Dev shape: (209760, 3)
Validation shape: (7999, 4)
Final train shape: (1887838, 3)
Final dev shape: (209760, 3)
Final validation shape: (7999, 4)

Preparing Datasets
Dataset({
    features: ['comment_text', 'label'],
    num_rows: 1887838
})
Dataset({
    features: ['comment_text', 'label'],
    num_rows: 209760
})
Dataset({
    features: ['comment_text', 'label', 'lang'],
    num_rows: 7999
})


In [ ]:
# ------------------------------------------------------------
# 6. Tokenizer and tokenization
# ------------------------------------------------------------
print_section("Loading Tokenizer")

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
dev_dataset = dev_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)





Loading Tokenizer


Map:   0%|          | 0/1887838 [00:00<?, ? examples/s]

Map:   0%|          | 0/209760 [00:00<?, ? examples/s]

Map:   0%|          | 0/7999 [00:00<?, ? examples/s]

In [ ]:
# ------------------------------------------------------------
# 7. Class weights for imbalanced training
# ------------------------------------------------------------
print_section("Computing Class Weights")

train_labels = train_df["label"].values
n_total = len(train_labels)
n_pos = np.sum(train_labels == 1)
n_neg = np.sum(train_labels == 0)

weight_for_0 = n_total / (2.0 * n_neg)
weight_for_1 = n_total / (2.0 * n_pos)

class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float)

print("Class weights:", class_weights.tolist())





Computing Class Weights
Class weights: [0.5445297956466675, 6.114217281341553]


In [ ]:
# ------------------------------------------------------------
# 8. Custom Trainer with weighted cross entropy
# ------------------------------------------------------------
print_section("Building Model")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss





Building Model


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# ------------------------------------------------------------
# 9. Training arguments
# ------------------------------------------------------------
print_section("Setting Training Arguments")

OUTPUT_DIR = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/mbert_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Core training setup
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,

    # Evaluation / saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,

    # Runtime
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)




`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



Setting Training Arguments


In [ ]:
# ------------------------------------------------------------
# 10. Build trainer
# ------------------------------------------------------------
print_section("Initializing Trainer")

trainer = WeightedCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)


# ------------------------------------------------------------
# 11. Train model
# ------------------------------------------------------------
print_section("Training mBERT")

trainer.train()

print("Training completed successfully.")





Initializing Trainer

Training mBERT


Epoch,Training Loss,Validation Loss
1,0.720922,0.690196
2,0.380825,0.594402


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Training completed successfully.


In [ ]:
# ------------------------------------------------------------
# 12. Predict on English Dev
# ------------------------------------------------------------
print_section("Predicting on English Dev Set")

dev_predictions = trainer.predict(dev_dataset)
dev_logits = dev_predictions.predictions

# Convert 2-class logits into probability of class 1
dev_prob = F.softmax(torch.tensor(dev_logits), dim=1).numpy()[:, 1]
y_dev = dev_df["label"].values

# Optional debug check
print("dev_logits summary:")
print("min:", np.min(dev_logits))
print("max:", np.max(dev_logits))
print("mean:", np.mean(dev_logits))
print("std:", np.std(dev_logits))

print("\ndev_prob summary:")
print("min:", np.min(dev_prob))
print("max:", np.max(dev_prob))
print("mean:", np.mean(dev_prob))
print("std:", np.std(dev_prob))

print("\nFirst 20 dev_prob:")
print(dev_prob[:20])

dev_metrics = evaluate_predictions(
    y_true=y_dev,
    y_prob=dev_prob,
    threshold=0.5
)

print_metrics(dev_metrics, "English Dev Set")

print_section("English Dev Set - Classification Report")
print(classification_report(y_dev, (dev_prob >= 0.5).astype(int), digits=4, zero_division=0))





Predicting on English Dev Set


dev_logits summary:
min: -2.6289062
max: 2.6914062
mean: 0.06321109
std: 2.2728796

dev_prob summary:
min: 0.0054692742
max: 0.98774695
mean: 0.10183974
std: 0.26043025

First 20 dev_prob:
[0.00569696 0.00573025 0.0085113  0.00552265 0.01994693 0.00637594
 0.00571913 0.02615562 0.98760426 0.00891613 0.25196967 0.00765103
 0.00717688 0.00900284 0.453761   0.01009132 0.00950408 0.00997492
 0.00932197 0.01264932]

English Dev Set - Evaluation Metrics
Threshold: 0.50
ROC-AUC: 0.8818
Precision: 0.6006
Recall: 0.6681
F1 Score: 0.6325
Positive Prediction Rate: 0.0910
Confusion Matrix:
[[184985   7622]
 [  5693  11460]]

English Dev Set - Classification Report
              precision    recall  f1-score   support

           0     0.9701    0.9604    0.9653    192607
           1     0.6006    0.6681    0.6325     17153

    accuracy                         0.9365    209760
   macro avg     0.7854    0.8143    0.7989    209760
weighted avg     0.9399    0.9365    0.9381    209760



In [ ]:
# ------------------------------------------------------------
# 13. Predict on multilingual validation
# ------------------------------------------------------------
print_section("Predicting on Multilingual Validation Set")

val_predictions = trainer.predict(val_dataset)
val_logits = val_predictions.predictions
val_prob = F.softmax(torch.tensor(val_logits), dim=1).numpy()[:, 1]
y_val = val_df["label"].values

val_metrics = evaluate_predictions(
    y_true=y_val,
    y_prob=val_prob,
    threshold=0.5
)

print_metrics(val_metrics, "Multilingual Validation Set")

print_section("Multilingual Validation Set - Classification Report")
print(classification_report(y_val, (val_prob >= 0.5).astype(int), digits=4, zero_division=0))





Predicting on Multilingual Validation Set



Multilingual Validation Set - Evaluation Metrics
Threshold: 0.50
ROC-AUC: 0.7259
Precision: 0.4150
Recall: 0.2480
F1 Score: 0.3104
Positive Prediction Rate: 0.0919
Confusion Matrix:
[[6339  430]
 [ 925  305]]

Multilingual Validation Set - Classification Report
              precision    recall  f1-score   support

           0     0.8727    0.9365    0.9034      6769
           1     0.4150    0.2480    0.3104      1230

    accuracy                         0.8306      7999
   macro avg     0.6438    0.5922    0.6069      7999
weighted avg     0.8023    0.8306    0.8123      7999



In [ ]:
# ------------------------------------------------------------
# 14. Per-language evaluation on validation set
# ------------------------------------------------------------
print_section("Per-Language Evaluation on Validation Set")

val_eval_df = val_df.copy()
val_eval_df["pred_prob"] = val_prob

per_lang_results = evaluate_by_language(
    df=val_eval_df,
    prob_col="pred_prob",
    label_col="toxic",
    lang_col="lang",
    threshold=0.5
)

print(per_lang_results)





Per-Language Evaluation on Validation Set
  lang  n_samples  positive_ratio   roc_auc  precision    recall        f1  \
0   es       2500        0.168800  0.758660   0.462783  0.338863  0.391245   
1   tr       2999        0.106702  0.727161   0.369231  0.150000  0.213333   
2   it       2500        0.195200  0.682291   0.385135  0.233607  0.290816   

   positive_prediction_rate  
0                  0.123600  
1                  0.043348  
2                  0.118400  


In [ ]:
# ------------------------------------------------------------
# 15. Save outputs for later modules
# ------------------------------------------------------------
print_section("Saving Outputs")

SAVE_MBERT_OUTPUTS = True
OUTPUT_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs"

if SAVE_MBERT_OUTPUTS:
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    # Save trained model
    trainer.save_model(f"{OUTPUT_PATH}/mbert_best_model")
    tokenizer.save_pretrained(f"{OUTPUT_PATH}/mbert_best_model")

    # Save dev predictions
    dev_output = dev_df.copy()
    dev_output["pred_prob_mbert"] = dev_prob
    dev_output["pred_label_mbert_05"] = (dev_prob >= 0.5).astype(int)
    dev_output.to_csv(f"{OUTPUT_PATH}/mbert_dev_predictions.csv", index=False)

    # Save validation predictions
    val_output = val_df.copy()
    val_output["pred_prob_mbert"] = val_prob
    val_output["pred_label_mbert_05"] = (val_prob >= 0.5).astype(int)
    val_output.to_csv(f"{OUTPUT_PATH}/mbert_validation_predictions.csv", index=False)

    # Save metrics
    metrics_dict = {
        "english_dev": dev_metrics,
        "multilingual_validation": val_metrics
    }

    with open(f"{OUTPUT_PATH}/mbert_metrics.json", "w") as f:
        json.dump(metrics_dict, f, indent=4)

    per_lang_results.to_csv(f"{OUTPUT_PATH}/mbert_per_language_results.csv", index=False)

    print(f"Outputs saved to: {OUTPUT_PATH}")





Saving Outputs


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Outputs saved to: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs


In [ ]:
# ------------------------------------------------------------
# 16. Final summary
# ------------------------------------------------------------
print_section("Module 4.3.1 Final Summary")

summary_table = pd.DataFrame([
    {
        "dataset": "english_dev",
        "roc_auc": round(dev_metrics["roc_auc"], 4),
        "precision": round(dev_metrics["precision"], 4),
        "recall": round(dev_metrics["recall"], 4),
        "f1": round(dev_metrics["f1"], 4),
        "positive_prediction_rate": round(dev_metrics["positive_prediction_rate"], 4)
    },
    {
        "dataset": "multilingual_validation",
        "roc_auc": round(val_metrics["roc_auc"], 4),
        "precision": round(val_metrics["precision"], 4),
        "recall": round(val_metrics["recall"], 4),
        "f1": round(val_metrics["f1"], 4),
        "positive_prediction_rate": round(val_metrics["positive_prediction_rate"], 4)
    }
])

print(summary_table)


Module 4.3.1 Final Summary
                   dataset  roc_auc  precision  recall      f1  \
0              english_dev   0.8818     0.6006  0.6681  0.6325   
1  multilingual_validation   0.7259     0.4150  0.2480  0.3104   

   positive_prediction_rate  
0                    0.0910  
1                    0.0919  


### 4.3.2 XLM-R (Please take the DEBUG version, ignore this one)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
def print_section(title: str) -> None:
    """Print a formatted section title."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def evaluate_predictions(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = 0.5
) -> dict:
    """
    Evaluate prediction probabilities using a fixed threshold.
    """
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "positive_prediction_rate": float(np.mean(y_pred)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }
    return metrics


def print_metrics(metrics: dict, dataset_name: str) -> None:
    """Pretty-print evaluation metrics."""
    print_section(f"{dataset_name} - Evaluation Metrics")
    print(f"Threshold: {metrics['threshold']:.2f}")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"Positive Prediction Rate: {metrics['positive_prediction_rate']:.4f}")
    print("Confusion Matrix:")
    print(np.array(metrics["confusion_matrix"]))


def evaluate_by_language(
    df: pd.DataFrame,
    prob_col: str = "pred_prob",
    label_col: str = "toxic",
    lang_col: str = "lang",
    threshold: float = 0.5
) -> pd.DataFrame:
    """
    Compute evaluation metrics for each language separately.
    """
    results = []

    for lang, group in df.groupby(lang_col):
        y_true = group[label_col].values
        y_prob = group[prob_col].values
        y_pred = (y_prob >= threshold).astype(int)

        result = {
            "lang": lang,
            "n_samples": len(group),
            "positive_ratio": float(group[label_col].mean()),
            "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "positive_prediction_rate": float(np.mean(y_pred))
        }
        results.append(result)

    return pd.DataFrame(results).sort_values(by="roc_auc", ascending=False).reset_index(drop=True)


def sigmoid_numpy(x: np.ndarray) -> np.ndarray:
    """Apply sigmoid to logits."""
    return 1.0 / (1.0 + np.exp(-x))


In [ ]:
# Load prepared data
print_section("Loading Prepared Data")

DATA_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/prepared data"

train_df = pd.read_csv(f"{DATA_PATH}/train_english_train.csv")
dev_df = pd.read_csv(f"{DATA_PATH}/train_english_dev.csv")
val_df = pd.read_csv(f"{DATA_PATH}/validation_clean.csv")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("Validation shape:", val_df.shape)

# Optional sampling for faster experimentation
USE_FAST_DEBUG = False

if USE_FAST_DEBUG:
    train_df = train_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)
    dev_df = dev_df.sample(frac=0.2, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(frac=0.2, random_state=SEED).reset_index(drop=True)

print("Final train shape:", train_df.shape)
print("Final dev shape:", dev_df.shape)
print("Final validation shape:", val_df.shape)


Loading Prepared Data
Train shape: (1887838, 3)
Dev shape: (209760, 3)
Validation shape: (7999, 4)
Final train shape: (1887838, 3)
Final dev shape: (209760, 3)
Final validation shape: (7999, 4)


In [ ]:
# Prepare Hugging Face datasets
print_section("Preparing Datasets")

train_df = train_df[["comment_text", "toxic"]].copy()
dev_df = dev_df[["comment_text", "toxic"]].copy()
val_df = val_df[["comment_text", "toxic", "lang"]].copy()

train_df["label"] = train_df["toxic"].astype(int)
dev_df["label"] = dev_df["toxic"].astype(int)
val_df["label"] = val_df["toxic"].astype(int)

train_dataset = Dataset.from_pandas(train_df[["comment_text", "label"]], preserve_index=False)
dev_dataset = Dataset.from_pandas(dev_df[["comment_text", "label"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["comment_text", "label", "lang"]], preserve_index=False)

print(train_dataset)
print(dev_dataset)
print(val_dataset)



Preparing Datasets
Dataset({
    features: ['comment_text', 'label'],
    num_rows: 1887838
})
Dataset({
    features: ['comment_text', 'label'],
    num_rows: 209760
})
Dataset({
    features: ['comment_text', 'label', 'lang'],
    num_rows: 7999
})


In [ ]:
# Tokenizer and tokenization
print_section("Loading Tokenizer")

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
dev_dataset = dev_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Loading Tokenizer


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1887838 [00:00<?, ? examples/s]

Map:   0%|          | 0/209760 [00:00<?, ? examples/s]

Map:   0%|          | 0/7999 [00:00<?, ? examples/s]

In [ ]:
# Class weights for imbalanced training
print_section("Computing Class Weights")

train_labels = train_df["label"].values
n_total = len(train_labels)
n_pos = np.sum(train_labels == 1)
n_neg = np.sum(train_labels == 0)

weight_for_0 = n_total / (2.0 * n_neg)
weight_for_1 = n_total / (2.0 * n_pos)

class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float)

print("Class weights:", class_weights.tolist())


Computing Class Weights
Class weights: [0.5445297956466675, 6.114217281341553]


In [ ]:
# Custom Trainer with weighted loss
print_section("Building Model")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="single_label_classification"
)

class WeightedBCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits.view(-1)

        loss_fn = torch.nn.BCEWithLogitsLoss(
            pos_weight=class_weights[1].to(logits.device)
        )
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


Building Model


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Training arguments
print_section("Setting Training Arguments")

OUTPUT_DIR = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/xlmr_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Core training setup
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,

    # Evaluation / saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,

    # Runtime
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



Setting Training Arguments


In [ ]:
# Build trainer
print_section("Initializing Trainer")

trainer = WeightedBCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)
# Train Model
print_section("Training XLM-R")
trainer.train()
print("Training completed successfully.")


Initializing Trainer

Training XLM-R


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Predict on English Dev
print_section("Predicting on English Dev Set")

dev_predictions = trainer.predict(dev_dataset)
dev_logits = dev_predictions.predictions.reshape(-1)
dev_prob = sigmoid_numpy(dev_logits)
y_dev = dev_df["label"].values

dev_metrics = evaluate_predictions(
    y_true=y_dev,
    y_prob=dev_prob,
    threshold=0.5
)

print_metrics(dev_metrics, "English Dev Set")

print_section("English Dev Set - Classification Report")
print(classification_report(y_dev, (dev_prob >= 0.5).astype(int), digits=4, zero_division=0))

In [ ]:
# Predict on multilingual validation
print_section("Predicting on Multilingual Validation Set")

val_predictions = trainer.predict(val_dataset)
val_logits = val_predictions.predictions.reshape(-1)
val_prob = sigmoid_numpy(val_logits)
y_val = val_df["label"].values

val_metrics = evaluate_predictions(
    y_true=y_val,
    y_prob=val_prob,
    threshold=0.5
)

print_metrics(val_metrics, "Multilingual Validation Set")

print_section("Multilingual Validation Set - Classification Report")
print(classification_report(y_val, (val_prob >= 0.5).astype(int), digits=4, zero_division=0))

In [ ]:
# Per-language evaluation on validation set
print_section("Per-Language Evaluation on Validation Set")

val_eval_df = val_df.copy()
val_eval_df["pred_prob"] = val_prob

per_lang_results = evaluate_by_language(
    df=val_eval_df,
    prob_col="pred_prob",
    label_col="toxic",
    lang_col="lang",
    threshold=0.5
)

print(per_lang_results)

In [ ]:
# Save outputs for later modules
print_section("Saving Outputs")

SAVE_XLMR_OUTPUTS = True
OUTPUT_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs"

if SAVE_XLMR_OUTPUTS:
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    # Save trained model and tokenizer
    trainer.save_model(f"{OUTPUT_PATH}/xlmr_best_model")
    tokenizer.save_pretrained(f"{OUTPUT_PATH}/xlmr_best_model")

    # Save dev predictions
    dev_output = dev_df.copy()
    dev_output["pred_prob_xlmr"] = dev_prob
    dev_output["pred_label_xlmr_05"] = (dev_prob >= 0.5).astype(int)
    dev_output.to_csv(f"{OUTPUT_PATH}/xlmr_dev_predictions.csv", index=False)

    # Save validation predictions
    val_output = val_df.copy()
    val_output["pred_prob_xlmr"] = val_prob
    val_output["pred_label_xlmr_05"] = (val_prob >= 0.5).astype(int)
    val_output.to_csv(f"{OUTPUT_PATH}/xlmr_validation_predictions.csv", index=False)

    # Save metrics
    metrics_dict = {
        "english_dev": dev_metrics,
        "multilingual_validation": val_metrics
    }

    with open(f"{OUTPUT_PATH}/xlmr_metrics.json", "w") as f:
        json.dump(metrics_dict, f, indent=4)

    per_lang_results.to_csv(f"{OUTPUT_PATH}/xlmr_per_language_results.csv", index=False)

    print(f"Outputs saved to: {OUTPUT_PATH}")

In [ ]:
# Final summary
print_section("Module 4.3.2 Final Summary")

summary_table = pd.DataFrame([
    {
        "dataset": "english_dev",
        "roc_auc": round(dev_metrics["roc_auc"], 4),
        "precision": round(dev_metrics["precision"], 4),
        "recall": round(dev_metrics["recall"], 4),
        "f1": round(dev_metrics["f1"], 4),
        "positive_prediction_rate": round(dev_metrics["positive_prediction_rate"], 4)
    },
    {
        "dataset": "multilingual_validation",
        "roc_auc": round(val_metrics["roc_auc"], 4),
        "precision": round(val_metrics["precision"], 4),
        "recall": round(val_metrics["recall"], 4),
        "f1": round(val_metrics["f1"], 4),
        "positive_prediction_rate": round(val_metrics["positive_prediction_rate"], 4)
    }
])

print(summary_table)

#### Debug

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Utility Functions
def print_section(title: str) -> None:
    """Print a formatted section title."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def evaluate_predictions(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = 0.5
) -> dict:
    """
    Evaluate prediction probabilities using a fixed threshold.
    """
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "positive_prediction_rate": float(np.mean(y_pred)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }
    return metrics


def print_metrics(metrics: dict, dataset_name: str) -> None:
    """Pretty-print evaluation metrics."""
    print_section(f"{dataset_name} - Evaluation Metrics")
    print(f"Threshold: {metrics['threshold']:.2f}")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"Positive Prediction Rate: {metrics['positive_prediction_rate']:.4f}")
    print("Confusion Matrix:")
    print(np.array(metrics["confusion_matrix"]))


def evaluate_by_language(
    df: pd.DataFrame,
    prob_col: str = "pred_prob",
    label_col: str = "toxic",
    lang_col: str = "lang",
    threshold: float = 0.5
) -> pd.DataFrame:
    """
    Compute evaluation metrics for each language separately.
    """
    results = []

    for lang, group in df.groupby(lang_col):
        y_true = group[label_col].values
        y_prob = group[prob_col].values
        y_pred = (y_prob >= threshold).astype(int)

        result = {
            "lang": lang,
            "n_samples": len(group),
            "positive_ratio": float(group[label_col].mean()),
            "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "positive_prediction_rate": float(np.mean(y_pred))
        }
        results.append(result)

    return pd.DataFrame(results).sort_values(by="roc_auc", ascending=False).reset_index(drop=True)


In [ ]:
# Load prepared data
print_section("Loading Prepared Data")

DATA_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/prepared data"

train_df = pd.read_csv(f"{DATA_PATH}/train_english_train.csv")
dev_df = pd.read_csv(f"{DATA_PATH}/train_english_dev.csv")
val_df = pd.read_csv(f"{DATA_PATH}/validation_clean.csv")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("Validation shape:", val_df.shape)

# Optional sampling for faster experimentation
USE_FAST_DEBUG = False

if USE_FAST_DEBUG:
    train_df = train_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)
    dev_df = dev_df.sample(frac=0.2, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(frac=0.2, random_state=SEED).reset_index(drop=True)

print("Final train shape:", train_df.shape)
print("Final dev shape:", dev_df.shape)
print("Final validation shape:", val_df.shape)

# Prepare Hugging Face datasets
print_section("Preparing Datasets")

train_df = train_df[["comment_text", "toxic"]].copy()
dev_df = dev_df[["comment_text", "toxic"]].copy()
val_df = val_df[["comment_text", "toxic", "lang"]].copy()

train_df["label"] = train_df["toxic"].astype(int)
dev_df["label"] = dev_df["toxic"].astype(int)
val_df["label"] = val_df["toxic"].astype(int)

train_dataset = Dataset.from_pandas(train_df[["comment_text", "label"]], preserve_index=False)
dev_dataset = Dataset.from_pandas(dev_df[["comment_text", "label"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["comment_text", "label", "lang"]], preserve_index=False)

print(train_dataset)
print(dev_dataset)
print(val_dataset)



Loading Prepared Data
Train shape: (1887838, 3)
Dev shape: (209760, 3)
Validation shape: (7999, 4)
Final train shape: (1887838, 3)
Final dev shape: (209760, 3)
Final validation shape: (7999, 4)

Preparing Datasets
Dataset({
    features: ['comment_text', 'label'],
    num_rows: 1887838
})
Dataset({
    features: ['comment_text', 'label'],
    num_rows: 209760
})
Dataset({
    features: ['comment_text', 'label', 'lang'],
    num_rows: 7999
})


In [ ]:
# Tokenizer and tokenization
print_section("Loading Tokenizer")

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
dev_dataset = dev_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Loading Tokenizer


Map:   0%|          | 0/1887838 [00:00<?, ? examples/s]

Map:   0%|          | 0/209760 [00:00<?, ? examples/s]

Map:   0%|          | 0/7999 [00:00<?, ? examples/s]

In [ ]:
# Class weights for imbalanced training
print_section("Computing Class Weights")

train_labels = train_df["label"].values
n_total = len(train_labels)
n_pos = np.sum(train_labels == 1)
n_neg = np.sum(train_labels == 0)

weight_for_0 = n_total / (2.0 * n_neg)
weight_for_1 = n_total / (2.0 * n_pos)

class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float)

print("Class weights:", class_weights.tolist())


Computing Class Weights
Class weights: [0.5445297956466675, 6.114217281341553]


In [ ]:
# Custom Trainer with weighted cross entropy
print_section("Building Model")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


Building Model


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Training arguments
print_section("Setting Training Arguments")

OUTPUT_DIR = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/xlmr_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Core training setup
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,

    # Evaluation / saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,

    # Runtime
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



Setting Training Arguments


In [ ]:
# Build trainer
print_section("Initializing Trainer")

trainer = WeightedCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# Train model
print_section("Training XLM-R")

trainer.train()

print("Training completed successfully.")



Initializing Trainer

Training XLM-R


Epoch,Training Loss,Validation Loss
1,0.858797,1.231775
2,0.718801,0.856097


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training completed successfully.


In [ ]:
# Predict on English Dev
print_section("Predicting on English Dev Set")

dev_predictions = trainer.predict(dev_dataset)
dev_logits = dev_predictions.predictions

# Convert 2-class logits into probability of class 1
dev_prob = F.softmax(torch.tensor(dev_logits), dim=1).numpy()[:, 1]
y_dev = dev_df["label"].values

# Optional debug check
print("dev_logits summary:")
print("min:", np.min(dev_logits))
print("max:", np.max(dev_logits))
print("mean:", np.mean(dev_logits))
print("std:", np.std(dev_logits))

print("\ndev_prob summary:")
print("min:", np.min(dev_prob))
print("max:", np.max(dev_prob))
print("mean:", np.mean(dev_prob))
print("std:", np.std(dev_prob))

print("\nFirst 20 dev_prob:")
print(dev_prob[:20])

dev_metrics = evaluate_predictions(
    y_true=y_dev,
    y_prob=dev_prob,
    threshold=0.5
)

print_metrics(dev_metrics, "English Dev Set")

print_section("English Dev Set - Classification Report")
print(classification_report(y_dev, (dev_prob >= 0.5).astype(int), digits=4, zero_division=0))



Predicting on English Dev Set


dev_logits summary:
min: -3.8027344
max: 3.265625
mean: -0.5003213
std: 2.8102088

dev_prob summary:
min: 0.0017996542
max: 0.99528164
mean: 0.06533993
std: 0.22402474

First 20 dev_prob:
[0.00613318 0.00364999 0.00209927 0.00576373 0.02771656 0.00192298
 0.01172115 0.01712732 0.99351174 0.02658183 0.03114383 0.00233223
 0.0021998  0.00482972 0.02824778 0.00192298 0.00295215 0.00217423
 0.00217423 0.00261634]

English Dev Set - Evaluation Metrics
Threshold: 0.50
ROC-AUC: 0.9035
Precision: 0.6885
Recall: 0.4907
F1 Score: 0.5730
Positive Prediction Rate: 0.0583
Confusion Matrix:
[[188799   3808]
 [  8736   8417]]

English Dev Set - Classification Report
              precision    recall  f1-score   support

           0     0.9558    0.9802    0.9678    192607
           1     0.6885    0.4907    0.5730     17153

    accuracy                         0.9402    209760
   macro avg     0.8221    0.7355    0.7704    209760
weighted avg     0.9339    0.9402    0.9356    209760



In [ ]:
# Predict on multilingual validation
print_section("Predicting on Multilingual Validation Set")

val_predictions = trainer.predict(val_dataset)
val_logits = val_predictions.predictions
val_prob = F.softmax(torch.tensor(val_logits), dim=1).numpy()[:, 1]
y_val = val_df["label"].values

val_metrics = evaluate_predictions(
    y_true=y_val,
    y_prob=val_prob,
    threshold=0.5
)

print_metrics(val_metrics, "Multilingual Validation Set")

print_section("Multilingual Validation Set - Classification Report")
print(classification_report(y_val, (val_prob >= 0.5).astype(int), digits=4, zero_division=0))


Predicting on Multilingual Validation Set



Multilingual Validation Set - Evaluation Metrics
Threshold: 0.50
ROC-AUC: 0.8363
Precision: 0.5060
Recall: 0.1715
F1 Score: 0.2562
Positive Prediction Rate: 0.0521
Confusion Matrix:
[[6563  206]
 [1019  211]]

Multilingual Validation Set - Classification Report
              precision    recall  f1-score   support

           0     0.8656    0.9696    0.9146      6769
           1     0.5060    0.1715    0.2562      1230

    accuracy                         0.8469      7999
   macro avg     0.6858    0.5706    0.5854      7999
weighted avg     0.8103    0.8469    0.8134      7999



In [ ]:
# Per-language evaluation on validation set
print_section("Per-Language Evaluation on Validation Set")

val_eval_df = val_df.copy()
val_eval_df["pred_prob"] = val_prob

per_lang_results = evaluate_by_language(
    df=val_eval_df,
    prob_col="pred_prob",
    label_col="toxic",
    lang_col="lang",
    threshold=0.5
)

print(per_lang_results)


Per-Language Evaluation on Validation Set
  lang  n_samples  positive_ratio   roc_auc  precision    recall        f1  \
0   tr       2999        0.106702  0.925873   0.950000  0.059375  0.111765   
1   es       2500        0.168800  0.813498   0.588235  0.284360  0.383387   
2   it       2500        0.195200  0.775337   0.373057  0.147541  0.211454   

   positive_prediction_rate  
0                  0.006669  
1                  0.081600  
2                  0.077200  


In [ ]:
# Save outputs for later modules
print_section("Saving Outputs")

SAVE_XLMR_OUTPUTS = True
OUTPUT_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs"

if SAVE_XLMR_OUTPUTS:
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    # Save trained model
    trainer.save_model(f"{OUTPUT_PATH}/xlmr_best_model")
    tokenizer.save_pretrained(f"{OUTPUT_PATH}/xlmr_best_model")

    # Save dev predictions
    dev_output = dev_df.copy()
    dev_output["pred_prob_xlmr"] = dev_prob
    dev_output["pred_label_xlmr_05"] = (dev_prob >= 0.5).astype(int)
    dev_output.to_csv(f"{OUTPUT_PATH}/xlmr_dev_predictions.csv", index=False)

    # Save validation predictions
    val_output = val_df.copy()
    val_output["pred_prob_xlmr"] = val_prob
    val_output["pred_label_xlmr_05"] = (val_prob >= 0.5).astype(int)
    val_output.to_csv(f"{OUTPUT_PATH}/xlmr_validation_predictions.csv", index=False)

    # Save metrics
    metrics_dict = {
        "english_dev": dev_metrics,
        "multilingual_validation": val_metrics
    }

    with open(f"{OUTPUT_PATH}/xlmr_metrics.json", "w") as f:
        json.dump(metrics_dict, f, indent=4)

    per_lang_results.to_csv(f"{OUTPUT_PATH}/xlmr_per_language_results.csv", index=False)

    print(f"Outputs saved to: {OUTPUT_PATH}")


Saving Outputs


NameError: name 'trainer' is not defined

In [ ]:
# Final summary
print_section("Module 4.3.2 Final Summary")

summary_table = pd.DataFrame([
    {
        "dataset": "english_dev",
        "roc_auc": round(dev_metrics["roc_auc"], 4),
        "precision": round(dev_metrics["precision"], 4),
        "recall": round(dev_metrics["recall"], 4),
        "f1": round(dev_metrics["f1"], 4),
        "positive_prediction_rate": round(dev_metrics["positive_prediction_rate"], 4)
    },
    {
        "dataset": "multilingual_validation",
        "roc_auc": round(val_metrics["roc_auc"], 4),
        "precision": round(val_metrics["precision"], 4),
        "recall": round(val_metrics["recall"], 4),
        "f1": round(val_metrics["f1"], 4),
        "positive_prediction_rate": round(val_metrics["positive_prediction_rate"], 4)
    }
])

print(summary_table)


Module 4.3.2 Final Summary
                   dataset  roc_auc  precision  recall      f1  \
0              english_dev   0.9035     0.6885  0.4907  0.5730   
1  multilingual_validation   0.8363     0.5060  0.1715  0.2562   

   positive_prediction_rate  
0                    0.0583  
1                    0.0521  


## 4.4 Evaluation and Comparison

In [ ]:
# Module 4.4: Model Comparison and Analysis
# Compare LR, BiLSTM, mBERT, and XLM-R under a unified setting

# Utility Functions
def print_section(title: str) -> None:
    """Print a formatted section title."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def evaluate_predictions(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = 0.5
) -> dict:
    """
    Evaluate binary classification results using probability outputs.
    """
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "threshold": threshold,
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "positive_prediction_rate": float(np.mean(y_pred)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }


def evaluate_by_language(
    df: pd.DataFrame,
    prob_col: str,
    label_col: str = "toxic",
    lang_col: str = "lang",
    threshold: float = 0.5
) -> pd.DataFrame:
    """
    Evaluate model performance by language.
    """
    rows = []

    for lang, group in df.groupby(lang_col):
        y_true = group[label_col].values
        y_prob = group[prob_col].values
        y_pred = (y_prob >= threshold).astype(int)

        row = {
            "lang": lang,
            "n_samples": len(group),
            "positive_ratio": float(group[label_col].mean()),
            "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "positive_prediction_rate": float(np.mean(y_pred))
        }
        rows.append(row)

    return pd.DataFrame(rows).sort_values("lang").reset_index(drop=True)


def build_overall_row(model_name: str, split_name: str, metrics: dict) -> dict:
    """
    Convert a metrics dictionary into one row for the summary table.
    """
    return {
        "model": model_name,
        "split": split_name,
        "roc_auc": round(metrics["roc_auc"], 4),
        "precision": round(metrics["precision"], 4),
        "recall": round(metrics["recall"], 4),
        "f1": round(metrics["f1"], 4),
        "positive_prediction_rate": round(metrics["positive_prediction_rate"], 4)
    }


def build_gap_table(dev_table: pd.DataFrame, val_table: pd.DataFrame) -> pd.DataFrame:
    """
    Build a table showing English dev vs multilingual validation gap.
    """
    merged = dev_table.merge(
        val_table,
        on="model",
        suffixes=("_dev", "_val")
    )

    merged["roc_auc_drop"] = merged["roc_auc_dev"] - merged["roc_auc_val"]
    merged["recall_drop"] = merged["recall_dev"] - merged["recall_val"]
    merged["f1_drop"] = merged["f1_dev"] - merged["f1_val"]

    selected_cols = [
        "model",
        "roc_auc_dev", "roc_auc_val", "roc_auc_drop",
        "recall_dev", "recall_val", "recall_drop",
        "f1_dev", "f1_val", "f1_drop"
    ]

    return merged[selected_cols].sort_values("roc_auc_val", ascending=False).reset_index(drop=True)


def compute_language_robustness(per_lang_df: pd.DataFrame, metric: str = "roc_auc") -> pd.DataFrame:
    """
    Summarize robustness across languages for each model.
    """
    summary_rows = []

    for model_name, group in per_lang_df.groupby("model"):
        valid_metric = group[metric].dropna()
        summary_rows.append({
            "model": model_name,
            f"{metric}_mean": round(valid_metric.mean(), 4) if len(valid_metric) > 0 else np.nan,
            f"{metric}_std": round(valid_metric.std(), 4) if len(valid_metric) > 0 else np.nan,
            f"{metric}_best": round(valid_metric.max(), 4) if len(valid_metric) > 0 else np.nan,
            f"{metric}_worst": round(valid_metric.min(), 4) if len(valid_metric) > 0 else np.nan,
            f"{metric}_gap_best_minus_worst": round(valid_metric.max() - valid_metric.min(), 4) if len(valid_metric) > 0 else np.nan
        })

    return pd.DataFrame(summary_rows).sort_values(f"{metric}_mean", ascending=False).reset_index(drop=True)

In [ ]:
# Paths and configuration
print_section("Setting Paths")

OUTPUT_PATH = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs"
THRESHOLD = 0.5

print("Output path:", OUTPUT_PATH)
print("Threshold used for comparison:", THRESHOLD)


Setting Paths
Output path: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs
Threshold used for comparison: 0.5


In [ ]:
# Load prediction files
# Expected files:
# - lr_dev_predictions.csv
# - lr_validation_predictions.csv
# - bilstm_dev_predictions.csv
# - bilstm_validation_predictions.csv
# - mbert_dev_predictions.csv
# - mbert_validation_predictions.csv
# - xlmr_dev_predictions.csv
# - xlmr_validation_predictions.csv

lr_dev = pd.read_csv(f"{OUTPUT_PATH}/lr_dev_predictions.csv")
lr_val = pd.read_csv(f"{OUTPUT_PATH}/lr_validation_predictions.csv")

bilstm_dev = pd.read_csv(f"{OUTPUT_PATH}/bilstm_dev_predictions.csv")
bilstm_val = pd.read_csv(f"{OUTPUT_PATH}/bilstm_validation_predictions.csv")

mbert_dev = pd.read_csv(f"{OUTPUT_PATH}/mbert_dev_predictions.csv")
mbert_val = pd.read_csv(f"{OUTPUT_PATH}/mbert_validation_predictions.csv")

xlmr_dev = pd.read_csv(f"{OUTPUT_PATH}/xlmr_dev_predictions.csv")
xlmr_val = pd.read_csv(f"{OUTPUT_PATH}/xlmr_validation_predictions.csv")

print("All prediction files loaded successfully.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/xlmr_dev_predictions.csv'

In [ ]:
# Define model file mappings
print_section("Defining Model Prediction Columns")

model_file_map = {
    "LR": {
        "dev_df": lr_dev,
        "val_df": lr_val,
        "dev_prob_col": "pred_prob_lr",
        "val_prob_col": "pred_prob_lr"
    },
    "BiLSTM": {
        "dev_df": bilstm_dev,
        "val_df": bilstm_val,
        "dev_prob_col": "pred_prob_bilstm",
        "val_prob_col": "pred_prob_bilstm"
    },
    "mBERT": {
        "dev_df": mbert_dev,
        "val_df": mbert_val,
        "dev_prob_col": "pred_prob_mbert",
        "val_prob_col": "pred_prob_mbert"
    },
    "XLM-R": {
        "dev_df": xlmr_dev,
        "val_df": xlmr_val,
        "dev_prob_col": "pred_prob_xlmr",
        "val_prob_col": "pred_prob_xlmr"
    }
}

print("Models included:", list(model_file_map.keys()))


Defining Model Prediction Columns
Models included: ['LR', 'BiLSTM', 'mBERT', 'XLM-R']


In [ ]:
# Compute overall metrics for each model
print_section("Computing Overall Comparison Metrics")

overall_rows = []
per_language_tables = []

for model_name, info in model_file_map.items():
    # Dev
    dev_df = info["dev_df"].copy()
    y_dev = dev_df["toxic"].astype(int).values
    dev_prob = dev_df[info["dev_prob_col"]].values

    dev_metrics = evaluate_predictions(
        y_true=y_dev,
        y_prob=dev_prob,
        threshold=THRESHOLD
    )

    overall_rows.append(build_overall_row(model_name, "english_dev", dev_metrics))

    # Validation
    val_df = info["val_df"].copy()
    y_val = val_df["toxic"].astype(int).values
    val_prob = val_df[info["val_prob_col"]].values

    val_metrics = evaluate_predictions(
        y_true=y_val,
        y_prob=val_prob,
        threshold=THRESHOLD
    )

    overall_rows.append(build_overall_row(model_name, "multilingual_validation", val_metrics))

    # Per-language
    per_lang_df = evaluate_by_language(
        df=val_df,
        prob_col=info["val_prob_col"],
        label_col="toxic",
        lang_col="lang",
        threshold=THRESHOLD
    )
    per_lang_df["model"] = model_name
    per_language_tables.append(per_lang_df)

overall_comparison = pd.DataFrame(overall_rows)
per_language_comparison = pd.concat(per_language_tables, axis=0).reset_index(drop=True)

print(overall_comparison)


Computing Overall Comparison Metrics
    model                    split  roc_auc  precision  recall      f1  \
0      LR              english_dev   0.9456     0.4616  0.8281  0.5927   
1      LR  multilingual_validation   0.6223     0.3924  0.0756  0.1268   
2  BiLSTM              english_dev   0.9482     0.3697  0.8968  0.5236   
3  BiLSTM  multilingual_validation   0.5679     0.2587  0.2057  0.2292   
4   mBERT              english_dev   0.8818     0.6006  0.6681  0.6325   
5   mBERT  multilingual_validation   0.7259     0.4150  0.2480  0.3104   
6   XLM-R              english_dev   0.9035     0.6885  0.4907  0.5730   
7   XLM-R  multilingual_validation   0.8363     0.5060  0.1715  0.2562   

   positive_prediction_rate  
0                    0.1467  
1                    0.0296  
2                    0.1984  
3                    0.1223  
4                    0.0910  
5                    0.0919  
6                    0.0583  
7                    0.0521  


In [ ]:
# Build comparison tables
print_section("Building Pivot Comparison Tables")

overall_pivot = overall_comparison.pivot(
    index="model",
    columns="split",
    values=["roc_auc", "precision", "recall", "f1", "positive_prediction_rate"]
)

overall_pivot = overall_pivot.sort_index()
print(overall_pivot)


Building Pivot Comparison Tables
           roc_auc                           precision  \
split  english_dev multilingual_validation english_dev   
model                                                    
BiLSTM      0.9482                  0.5679      0.3697   
LR          0.9456                  0.6223      0.4616   
XLM-R       0.9035                  0.8363      0.6885   
mBERT       0.8818                  0.7259      0.6006   

                                    recall                          \
split  multilingual_validation english_dev multilingual_validation   
model                                                                
BiLSTM                  0.2587      0.8968                  0.2057   
LR                      0.3924      0.8281                  0.0756   
XLM-R                   0.5060      0.4907                  0.1715   
mBERT                   0.4150      0.6681                  0.2480   

                f1                         positive_prediction_rate 

In [ ]:
# English dev vs multilingual validation gap
print_section("Computing Dev vs Validation Gap")

dev_table = overall_comparison[overall_comparison["split"] == "english_dev"].drop(columns=["split"]).reset_index(drop=True)
val_table = overall_comparison[overall_comparison["split"] == "multilingual_validation"].drop(columns=["split"]).reset_index(drop=True)

gap_table = build_gap_table(dev_table, val_table)
print(gap_table)


Computing Dev vs Validation Gap
    model  roc_auc_dev  roc_auc_val  roc_auc_drop  recall_dev  recall_val  \
0   XLM-R       0.9035       0.8363        0.0672      0.4907      0.1715   
1   mBERT       0.8818       0.7259        0.1559      0.6681      0.2480   
2      LR       0.9456       0.6223        0.3233      0.8281      0.0756   
3  BiLSTM       0.9482       0.5679        0.3803      0.8968      0.2057   

   recall_drop  f1_dev  f1_val  f1_drop  
0       0.3192  0.5730  0.2562   0.3168  
1       0.4201  0.6325  0.3104   0.3221  
2       0.7525  0.5927  0.1268   0.4659  
3       0.6911  0.5236  0.2292   0.2944  


In [ ]:
# Best model by metric
print_section("Best Model by Validation Metric")

val_only = overall_comparison[overall_comparison["split"] == "multilingual_validation"].copy()

best_auc_row = val_only.sort_values("roc_auc", ascending=False).iloc[0]
best_recall_row = val_only.sort_values("recall", ascending=False).iloc[0]
best_f1_row = val_only.sort_values("f1", ascending=False).iloc[0]

best_models_table = pd.DataFrame([
    {
        "criterion": "Best Validation ROC-AUC",
        "model": best_auc_row["model"],
        "value": best_auc_row["roc_auc"]
    },
    {
        "criterion": "Best Validation Recall",
        "model": best_recall_row["model"],
        "value": best_recall_row["recall"]
    },
    {
        "criterion": "Best Validation F1",
        "model": best_f1_row["model"],
        "value": best_f1_row["f1"]
    }
])

print(best_models_table)


Best Model by Validation Metric
                 criterion  model   value
0  Best Validation ROC-AUC  XLM-R  0.8363
1   Best Validation Recall  mBERT  0.2480
2       Best Validation F1  mBERT  0.3104


In [ ]:
# Per-language comparison tables
print_section("Per-Language Comparison Tables")

per_lang_auc_pivot = per_language_comparison.pivot(
    index="lang",
    columns="model",
    values="roc_auc"
).sort_index()

per_lang_recall_pivot = per_language_comparison.pivot(
    index="lang",
    columns="model",
    values="recall"
).sort_index()

per_lang_f1_pivot = per_language_comparison.pivot(
    index="lang",
    columns="model",
    values="f1"
).sort_index()

print("Per-language ROC-AUC:")
print(per_lang_auc_pivot)

print("\nPer-language Recall:")
print(per_lang_recall_pivot)

print("\nPer-language F1:")
print(per_lang_f1_pivot)


Per-Language Comparison Tables
Per-language ROC-AUC:
model    BiLSTM        LR     XLM-R     mBERT
lang                                         
es     0.641660  0.586452  0.813498  0.758660
it     0.524580  0.615092  0.775337  0.682291
tr     0.557773  0.660835  0.925873  0.727161

Per-language Recall:
model    BiLSTM        LR     XLM-R     mBERT
lang                                         
es     0.317536  0.113744  0.284360  0.338863
it     0.176230  0.034836  0.147541  0.233607
tr     0.103125  0.087500  0.059375  0.150000

Per-language F1:
model    BiLSTM        LR     XLM-R     mBERT
lang                                         
es     0.289730  0.169912  0.383387  0.391245
it     0.202830  0.063910  0.211454  0.290816
tr     0.151724  0.151351  0.111765  0.213333


In [ ]:
# Language robustness analysis
print_section("Language Robustness Analysis")

robustness_auc = compute_language_robustness(per_language_comparison, metric="roc_auc")
robustness_recall = compute_language_robustness(per_language_comparison, metric="recall")
robustness_f1 = compute_language_robustness(per_language_comparison, metric="f1")

print("ROC-AUC Robustness:")
print(robustness_auc)

print("\nRecall Robustness:")
print(robustness_recall)

print("\nF1 Robustness:")
print(robustness_f1)


Language Robustness Analysis
ROC-AUC Robustness:
    model  roc_auc_mean  roc_auc_std  roc_auc_best  roc_auc_worst  \
0   XLM-R        0.8382       0.0783        0.9259         0.7753   
1   mBERT        0.7227       0.0384        0.7587         0.6823   
2      LR        0.6208       0.0375        0.6608         0.5865   
3  BiLSTM        0.5747       0.0603        0.6417         0.5246   

   roc_auc_gap_best_minus_worst  
0                        0.1505  
1                        0.0764  
2                        0.0744  
3                        0.1171  

Recall Robustness:
    model  recall_mean  recall_std  recall_best  recall_worst  \
0   mBERT       0.2408      0.0946       0.3389        0.1500   
1  BiLSTM       0.1990      0.1090       0.3175        0.1031   
2   XLM-R       0.1638      0.1134       0.2844        0.0594   
3      LR       0.0787      0.0402       0.1137        0.0348   

   recall_gap_best_minus_worst  
0                       0.1889  
1                     

In [ ]:
# Simple ranking tables
print_section("Validation Ranking Tables")

val_ranking_auc = val_only[["model", "roc_auc"]].sort_values("roc_auc", ascending=False).reset_index(drop=True)
val_ranking_recall = val_only[["model", "recall"]].sort_values("recall", ascending=False).reset_index(drop=True)
val_ranking_f1 = val_only[["model", "f1"]].sort_values("f1", ascending=False).reset_index(drop=True)

print("Validation ROC-AUC Ranking:")
print(val_ranking_auc)

print("\nValidation Recall Ranking:")
print(val_ranking_recall)

print("\nValidation F1 Ranking:")
print(val_ranking_f1)


Validation Ranking Tables
Validation ROC-AUC Ranking:
    model  roc_auc
0   XLM-R   0.8363
1   mBERT   0.7259
2      LR   0.6223
3  BiLSTM   0.5679

Validation Recall Ranking:
    model  recall
0   mBERT  0.2480
1  BiLSTM  0.2057
2   XLM-R  0.1715
3      LR  0.0756

Validation F1 Ranking:
    model      f1
0   mBERT  0.3104
1   XLM-R  0.2562
2  BiLSTM  0.2292
3      LR  0.1268


In [ ]:
# Save outputs
print_section("Saving Comparison Outputs")

SAVE_COMPARISON_OUTPUTS = True
COMPARISON_PATH = f"{OUTPUT_PATH}/comparison_outputs"

if SAVE_COMPARISON_OUTPUTS:
    os.makedirs(COMPARISON_PATH, exist_ok=True)

    overall_comparison.to_csv(f"{COMPARISON_PATH}/overall_comparison_long.csv", index=False)
    overall_pivot.to_csv(f"{COMPARISON_PATH}/overall_comparison_pivot.csv")
    gap_table.to_csv(f"{COMPARISON_PATH}/dev_vs_validation_gap.csv", index=False)

    best_models_table.to_csv(f"{COMPARISON_PATH}/best_models_table.csv", index=False)

    per_language_comparison.to_csv(f"{COMPARISON_PATH}/per_language_comparison_long.csv", index=False)
    per_lang_auc_pivot.to_csv(f"{COMPARISON_PATH}/per_language_auc_pivot.csv")
    per_lang_recall_pivot.to_csv(f"{COMPARISON_PATH}/per_language_recall_pivot.csv")
    per_lang_f1_pivot.to_csv(f"{COMPARISON_PATH}/per_language_f1_pivot.csv")

    robustness_auc.to_csv(f"{COMPARISON_PATH}/robustness_auc.csv", index=False)
    robustness_recall.to_csv(f"{COMPARISON_PATH}/robustness_recall.csv", index=False)
    robustness_f1.to_csv(f"{COMPARISON_PATH}/robustness_f1.csv", index=False)

    val_ranking_auc.to_csv(f"{COMPARISON_PATH}/validation_auc_ranking.csv", index=False)
    val_ranking_recall.to_csv(f"{COMPARISON_PATH}/validation_recall_ranking.csv", index=False)
    val_ranking_f1.to_csv(f"{COMPARISON_PATH}/validation_f1_ranking.csv", index=False)

    print(f"Comparison outputs saved to: {COMPARISON_PATH}")


Saving Comparison Outputs
Comparison outputs saved to: /content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification/module4_outputs/comparison_outputs


In [ ]:
# Final summary for quick reading
print_section("Module 4.4 Final Summary")

final_summary = val_only[[
    "model",
    "roc_auc",
    "precision",
    "recall",
    "f1",
    "positive_prediction_rate"
]].sort_values("roc_auc", ascending=False).reset_index(drop=True)

print(final_summary)


Module 4.4 Final Summary
    model  roc_auc  precision  recall      f1  positive_prediction_rate
0   XLM-R   0.8363     0.5060  0.1715  0.2562                    0.0521
1   mBERT   0.7259     0.4150  0.2480  0.3104                    0.0919
2      LR   0.6223     0.3924  0.0756  0.1268                    0.0296
3  BiLSTM   0.5679     0.2587  0.2057  0.2292                    0.1223


## Temp: Termination

In [ ]:
# Temp Termination
PROJECT_ROOT = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification"
OUTPUT_PATH = f"{PROJECT_ROOT}/module4_outputs"
COMPARISON_PATH = f"{OUTPUT_PATH}/comparison_outputs"
STATE_DIR = f"{PROJECT_ROOT}/project_state"
STATE_FILE = f"{STATE_DIR}/run_state.json"

os.makedirs(STATE_DIR, exist_ok=True)

def load_state():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, "r") as f:
            return json.load(f)
    return {}

def save_state(state: dict):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=4)

required_files = [
    # 4.1 LR
    f"{OUTPUT_PATH}/lr_dev_predictions.csv",
    f"{OUTPUT_PATH}/lr_validation_predictions.csv",
    f"{OUTPUT_PATH}/lr_metrics.json",
    f"{OUTPUT_PATH}/lr_per_language_results.csv",

    # 4.2 BiLSTM
    f"{OUTPUT_PATH}/bilstm_dev_predictions.csv",
    f"{OUTPUT_PATH}/bilstm_validation_predictions.csv",
    f"{OUTPUT_PATH}/bilstm_metrics.json",
    f"{OUTPUT_PATH}/bilstm_per_language_results.csv",

    # 4.3.1 mBERT
    f"{OUTPUT_PATH}/mbert_dev_predictions.csv",
    f"{OUTPUT_PATH}/mbert_validation_predictions.csv",
    f"{OUTPUT_PATH}/mbert_metrics.json",
    f"{OUTPUT_PATH}/mbert_per_language_results.csv",
    f"{OUTPUT_PATH}/mbert_best_model/config.json",

    # 4.3.2 XLM-R
    f"{OUTPUT_PATH}/xlmr_dev_predictions.csv",
    f"{OUTPUT_PATH}/xlmr_validation_predictions.csv",
    f"{OUTPUT_PATH}/xlmr_metrics.json",
    f"{OUTPUT_PATH}/xlmr_per_language_results.csv",
    f"{OUTPUT_PATH}/xlmr_best_model/config.json",

    # 4.4 Comparison
    f"{COMPARISON_PATH}/overall_comparison_long.csv",
    f"{COMPARISON_PATH}/overall_comparison_pivot.csv",
    f"{COMPARISON_PATH}/dev_vs_validation_gap.csv",
    f"{COMPARISON_PATH}/best_models_table.csv",
    f"{COMPARISON_PATH}/per_language_comparison_long.csv",
    f"{COMPARISON_PATH}/validation_auc_ranking.csv",
    f"{COMPARISON_PATH}/validation_recall_ranking.csv",
    f"{COMPARISON_PATH}/validation_f1_ranking.csv"
]

missing_files = [file for file in required_files if not os.path.exists(file)]

if len(missing_files) > 0:
    print("The following required files are missing:")
    for file in missing_files:
        print(file)
    raise FileNotFoundError("Module 4 is not fully saved. Runtime will NOT shut down.")

state = load_state()
state["module_4_1_lr"] = {"status": "done"}
state["module_4_2_bilstm"] = {"status": "done"}
state["module_4_3_1_mbert"] = {"status": "done"}
state["module_4_3_2_xlmr"] = {"status": "done"}
state["module_4_4_comparison"] = {"status": "done"}
state["module_4_all_done"] = {
    "status": "done",
    "next_step": "module_5_threshold_optimization"
}
save_state(state)

print("All Module 4 outputs are confirmed saved.")
print("State file updated successfully.")
print("Next step: Module 5 Threshold Optimization")
print("Shutting down runtime now...")

# os.kill(os.getpid(), 9)

All Module 4 outputs are confirmed saved.
State file updated successfully.
Next step: Module 5 Threshold Optimization
Shutting down runtime now...


In [ ]:
# Temp: Resume
from google.colab import drive
drive.mount('/content/drive')

import json
import os

PROJECT_ROOT = "/content/drive/My Drive/BT5153 GP/Group project/jigsaw multilingual toxic comment classification"
STATE_FILE = f"{PROJECT_ROOT}/project_state/run_state.json"

if os.path.exists(STATE_FILE):
    with open(STATE_FILE, "r") as f:
        state = json.load(f)
    print(json.dumps(state, indent=4))
else:
    print("No state file found.")

Mounted at /content/drive
{
    "module_4_1_lr": {
        "status": "done"
    },
    "module_4_2_bilstm": {
        "status": "done"
    },
    "module_4_3_1_mbert": {
        "status": "done"
    },
    "module_4_3_2_xlmr": {
        "status": "done"
    },
    "module_4_4_comparison": {
        "status": "done"
    },
    "module_4_all_done": {
        "status": "done",
        "next_step": "module_5_threshold_optimization"
    }
}
